# Real Time Object Recognition System

My project for detecting everyday objects (person, laptop, bottle, phone, etc) in images and live video.

The plan:
- understand how object detection works
- get some test images and explore them
- load a pretrained YOLOv8 model
- detect objects in images
- (separate scripts) realtime webcam detection + streamlit app

I am using the pretrained yolov8n model (the "nano" version) which was trained on the COCO dataset. COCO has 80 everyday classes so it already knows how to find things like people, bottles, laptops etc without me training anything.

## Phase 1: how object recognition works

**Image classification** answers "what is in this image?" but only gives one label for the whole image.

**Object detection** answers "what is in this image AND where?" - it gives a box, a class and a confidence score for every object. That is what I need here.

Why YOLO:
- YOLO = You Only Look Once. Older methods slid a window over the image thousands of times. YOLO looks at the whole image one time through the network and predicts all the boxes at once, which is why it is fast enough for realtime video.
- It splits the image into a grid, and each grid cell predicts boxes + class probabilities.
- **confidence score**: how sure the model is that the box really contains that object (0 to 1). I filter out everything below a threshold (usually 0.4) because low confidence predictions are mostly wrong or duplicates.
- **IoU (intersection over union)**: used to remove duplicate boxes on the same object (non max suppression).

yolov8n is the smallest version, so it runs ok on cpu which is what I have.

## Phase 2: get some test images

I am using two classic test images from the ultralytics repo (a street scene with people and a bus, and a group of people) plus I can add more images later. These cover several COCO classes so they are good for testing.

In [ ]:
import os
import urllib.request

# this works the same on kaggle and locally
os.makedirs("dataset", exist_ok=True)

base = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
for name in ["bus.jpg", "zidane.jpg"]:
    path = f"dataset/{name}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(base + name, path)
        print("downloaded", name)

print(os.listdir("dataset"))

In [ ]:
# basic exploration, check the size and color format of the images
import cv2

img = cv2.imread("dataset/bus.jpg")
print("shape (h, w, channels):", img.shape)
print("dtype:", img.dtype)

# opencv loads colors as BGR not RGB, worth remembering
print("first pixel BGR:", img[0, 0])

In [ ]:
# showing with matplotlib, need to flip BGR -> RGB or the colors look wrong
import matplotlib.pyplot as plt

bus = cv2.imread("dataset/bus.jpg")
zidane = cv2.imread("dataset/zidane.jpg")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(bus[:, :, ::-1])
plt.title("bus.jpg")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(zidane[:, :, ::-1])
plt.title("zidane.jpg")
plt.axis("off")
plt.show()

## Phase 3: load the pretrained YOLO model

In [ ]:
# install ultralytics, -q to keep the output short
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

# first run downloads yolov8n.pt automatically (~6 mb)
model = YOLO("yolov8n.pt")
print(type(model))

In [ ]:
# the model knows 80 coco classes, these are the first 10
print(len(model.names), "classes")
print(model.names[:10])

# some ones I care about for my project
for want in ["person", "laptop", "bottle", "book", "cell phone", "chair"]:
    ids = [k for k, v in model.names.items() if v == want]
    print(want, "-> class id", ids)

## Phase 4: detect objects in images

In [ ]:
# run detection on the bus image
res = model("dataset/bus.jpg", conf=0.4)[0]

print(f"found {len(res.boxes)} objects")
for b in res.boxes:
    cls = int(b.cls[0])       # class id
    conf = float(b.conf[0])   # confidence
    xyxy = b.xyxy[0].tolist() # box corners x1,y1,x2,y2
    print(f"{model.names[cls]:<12} conf={conf:.2f}  box={[round(v) for v in xyxy]}")

In [ ]:
# drawing the boxes myself with cv2 instead of using the built in plot(),
# so I understand exactly what is being drawn
img = cv2.imread("dataset/bus.jpg")

for b in res.boxes:
    x1, y1, x2, y2 = map(int, b.xyxy[0])
    cls = int(b.cls[0])
    conf = float(b.conf[0])
    label = f"{model.names[cls]} {conf:.2f}"

    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(img, label, (x1, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 200, 0), 2)

os.makedirs("dataset/results", exist_ok=True)
cv2.imwrite("dataset/results/bus_detected.jpg", img)
print("saved dataset/results/bus_detected.jpg")

In [ ]:
# side by side before/after
orig = cv2.imread("dataset/bus.jpg")
detected = cv2.imread("dataset/results/bus_detected.jpg")

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.imshow(orig[:, :, ::-1])
plt.title("original")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(detected[:, :, ::-1])
plt.title("after detection")
plt.axis("off")
plt.show()

In [ ]:
# run it on every image in the dataset folder
import time

for name in sorted(os.listdir("dataset")):
    if not name.endswith(".jpg"):
        continue
    path = os.path.join("dataset", name)
    t0 = time.time()
    r = model(path, conf=0.4)[0]
    dt = time.time() - t0
    found = [model.names[int(b.cls[0])] for b in r.boxes]
    print(f"{name:<12} {dt:.2f}s  {len(found)} objects -> {found}")

## What I learned

- the difference between classification and detection
- how to load a pretrained yolov8 model and what the COCO classes are
- the result object gives boxes with class ids, confidences and coordinates
- opencv loads BGR so I have to flip channels before matplotlib
- drawing boxes manually is just rectangle + puttext

Next: realtime webcam detection and a streamlit app (the app.py and real_time.py scripts in the repo).